# 🌍 Seismic Attributes from Scratch
## MSc Applied Geophysics — Jupyter Notebook Project
### Building Envelope · Instantaneous Phase · Instantaneous Frequency · RMS Energy · 2D Section Attributes

---
> **Author:** MSc Applied Geophysics Student  
> **Topic:** Signal Processing for Seismic Attributes (No built-in libraries — NumPy only)  
> **Tools:** Python · NumPy · Matplotlib · SciPy (validation only)

---


---
# PART 1: Understanding Seismic Trace as Signal

---
## 🧠 Theory Cell 1.1 — What is a Seismic Trace?

**Intuition first:**

Imagine you hit the ground with a hammer. The energy travels down, bounces off rock layers underground, and comes back up. A geophone (sensor) on the surface records this returning energy over time. That recording — a simple list of numbers varying with time — is called a **seismic trace**.

Each number in the trace represents the **ground displacement** (or velocity, or acceleration) at a specific moment in time. So a seismic trace is fundamentally just a **1D time series signal**.

**Key properties of a seismic trace:**

| Property | Symbol | Meaning |
|---|---|---|
| Sampling interval | `dt` | Time between two consecutive samples (e.g., 2 ms) |
| Number of samples | `N` | Total length of the trace |
| Total time | `T = N × dt` | Duration of recording |
| Sampling frequency | `fs = 1/dt` | How many samples per second |
| Nyquist frequency | `fn = fs/2` | Maximum frequency we can reliably represent |


---
## 📐 Mathematical Formulation Cell 1.2

A seismic trace is represented as a **discrete time series**:

$$x[n] = x(n \cdot \Delta t), \quad n = 0, 1, 2, \ldots, N-1$$

**Nyquist–Shannon Sampling Theorem:**

$$f_{Nyquist} = \frac{f_s}{2} = \frac{1}{2 \Delta t}$$

**Convolutional model of seismics:**

$$x(t) = w(t) * r(t)$$

Where:
- $w(t)$ → **wavelet** (e.g., Ricker wavelet)
- $r(t)$ → **reflectivity series** (spikes at geological interfaces)
- $*$ → convolution operator


In [ ]:
# ============================================================
# PART 1: Synthetic Seismic Trace Construction
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert  # ONLY used in validation cells

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f', 'axes.facecolor': '#1a1a1a',
    'axes.edgecolor': '#444', 'axes.labelcolor': 'white',
    'xtick.color': 'white', 'ytick.color': 'white',
    'text.color': 'white', 'grid.color': '#333',
    'grid.linestyle': '--', 'grid.alpha': 0.5, 'figure.dpi': 110
})

# ── Time axis ──
dt    = 0.002          # Sampling interval: 2 ms
t_max = 1.0            # Total recording time: 1 second
t     = np.arange(0, t_max, dt)
N     = len(t)
fs    = 1.0 / dt
f_nyquist = fs / 2

print("=" * 50)
print("  SEISMIC TRACE PARAMETERS")
print("=" * 50)
print(f"  Sampling interval (dt)   : {dt*1000:.1f} ms")
print(f"  Total samples (N)        : {N}")
print(f"  Total time (T)           : {t_max:.2f} s")
print(f"  Sampling frequency (fs)  : {fs:.0f} Hz")
print(f"  Nyquist frequency        : {f_nyquist:.0f} Hz")
print("=" * 50)


In [ ]:
# ── Ricker Wavelet from scratch ──
def ricker_wavelet(t_wav, f_peak):
    """
    Ricker (Mexican hat) wavelet.
    w(t) = (1 - 2π²f²t²) * exp(-π²f²t²)
    """
    pi2 = (np.pi * f_peak * t_wav) ** 2
    return (1 - 2 * pi2) * np.exp(-pi2)

t_wav   = np.arange(-0.1, 0.1, dt)
f_peak  = 30
wavelet = ricker_wavelet(t_wav, f_peak)

# ── Reflectivity series ──
reflectivity = np.zeros(N)
reflector_times      = [0.20, 0.40, 0.55, 0.75, 0.88]
reflector_amplitudes = [ 0.8, -0.6,  1.0, -0.4,  0.7]
for t_ref, amp in zip(reflector_times, reflector_amplitudes):
    reflectivity[int(t_ref / dt)] = amp

# ── Convolve: x(t) = w(t) * r(t) ──
seismic_trace = np.convolve(reflectivity, wavelet, mode='same')

# ── Add noise ──
np.random.seed(42)
noise_level = 0.05
noise = noise_level * np.random.randn(N)
seismic_trace_noisy = seismic_trace + noise

print(f"Wavelet samples   : {len(t_wav)}")
print(f"Reflectors        : {len(reflector_times)}")
print(f"Trace length      : {len(seismic_trace)} samples")
print(f"Amplitude range   : [{seismic_trace.min():.3f}, {seismic_trace.max():.3f}]")


## 📊 Visualization Cell 1.3 — Seismic Trace Components

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12))
fig.suptitle('PART 1 — Seismic Trace Construction',
             fontsize=15, fontweight='bold', color='white', y=1.01)

# Wavelet
ax = axes[0]
ax.plot(t_wav*1000, wavelet, color='#00BFFF', lw=2)
ax.fill_between(t_wav*1000, wavelet, 0, where=(wavelet>0), alpha=0.3, color='#00BFFF')
ax.fill_between(t_wav*1000, wavelet, 0, where=(wavelet<0), alpha=0.3, color='#FF6347')
ax.axhline(0, color='#666', lw=0.8)
ax.set_title(f'Ricker Wavelet  (f_peak = {f_peak} Hz)', color='white')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Amplitude'); ax.grid(True)

# Reflectivity
ax = axes[1]
for t_ref, amp in zip(reflector_times, reflector_amplitudes):
    c = '#00FF7F' if amp > 0 else '#FF4500'
    ax.vlines(t_ref*1000, 0, amp, color=c, lw=2.5)
    ax.plot(t_ref*1000, amp, 'o', color=c, ms=7)
ax.axhline(0, color='#666', lw=0.8)
ax.set_title('Reflectivity Series', color='white')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Refl. Coeff.'); ax.grid(True)

# Clean trace
ax = axes[2]
ax.plot(t*1000, seismic_trace, color='#FFD700', lw=1.5, label='Clean trace')
ax.fill_between(t*1000, seismic_trace, 0, where=(seismic_trace>0), alpha=0.25, color='#FFD700')
ax.fill_between(t*1000, seismic_trace, 0, where=(seismic_trace<0), alpha=0.25, color='#FF6347')
ax.axhline(0, color='#666', lw=0.8)
ax.set_title('Clean Seismic Trace  (w(t) ∗ r(t))', color='white')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Amplitude'); ax.grid(True)

# Noisy trace
ax = axes[3]
ax.plot(t*1000, seismic_trace_noisy, color='#B0C4DE', lw=1, alpha=0.85, label='Noisy trace')
ax.plot(t*1000, seismic_trace, color='#FFD700', lw=1.5, alpha=0.6, ls='--', label='Clean (ref)')
ax.axhline(0, color='#666', lw=0.8)
ax.set_title('Noisy Seismic Trace', color='white')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Amplitude'); ax.grid(True); ax.legend(fontsize=9)

plt.tight_layout(); plt.show()


## 📊 Visualization Cell 1.4 — Frequency Content

In [ ]:
def compute_spectrum(signal, dt):
    N = len(signal)
    fft_vals = np.fft.fft(signal)
    freqs_full = np.fft.fftfreq(N, d=dt)
    amp_full = np.abs(fft_vals) / N
    pos = freqs_full >= 0
    return freqs_full[pos], amp_full[pos] * 2

freqs_tr, amp_tr = compute_spectrum(seismic_trace, dt)
freqs_n,  amp_n  = compute_spectrum(seismic_trace_noisy, dt)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('PART 1 — Frequency Content', fontsize=14, fontweight='bold', color='white')

axes[0].plot(t*1000, seismic_trace, color='#FFD700', lw=1.5, label='Clean')
axes[0].plot(t*1000, seismic_trace_noisy, color='#B0C4DE', lw=0.8, alpha=0.7, label='Noisy')
axes[0].axhline(0, color='#555', lw=0.8)
axes[0].set(title='Time Domain', xlabel='Time (ms)', ylabel='Amplitude')
axes[0].legend(fontsize=9); axes[0].grid(True)

axes[1].plot(freqs_tr, amp_tr, color='#FFD700', lw=2, label='Clean spectrum')
axes[1].plot(freqs_n,  amp_n,  color='#B0C4DE', lw=1, alpha=0.7, label='Noisy spectrum')
axes[1].axvline(f_peak, color='#00FF7F', lw=1.5, ls='--', label=f'Peak = {f_peak} Hz')
axes[1].axvline(f_nyquist, color='#FF4500', lw=1.5, ls='--', label=f'Nyquist = {f_nyquist:.0f} Hz')
axes[1].set(title='Amplitude Spectrum', xlabel='Frequency (Hz)', ylabel='Amplitude', xlim=[0,150])
axes[1].legend(fontsize=9); axes[1].grid(True)

plt.tight_layout(); plt.show()


## 🔍 Validation Cell 1.5

In [ ]:
dominant_freq = freqs_tr[np.argmax(amp_tr)]
trace_energy  = np.sum(seismic_trace**2)
noise_energy  = np.sum(noise**2)
snr = 10 * np.log10(trace_energy / noise_energy)

print("=" * 55)
print("  PART 1 — VALIDATION SUMMARY")
print("=" * 55)
print(f"  Dominant frequency : {dominant_freq:.1f} Hz  (expected {f_peak} Hz)")
print(f"  Nyquist OK         : {f_nyquist:.0f} Hz > {f_peak} Hz  ✓")
print(f"  SNR                : {snr:.2f} dB")
print(f"  Trace length OK    : {len(seismic_trace)} == {N}  ✓")
print("=" * 55)
print("  ✅ PART 1 COMPLETE")
print("=" * 55)


---
# PART 2: Hilbert Transform (CORE PART)

---
## 🧠 Theory Cell 2.1 — Intuition of the Hilbert Transform

**The problem:** A seismic trace $x(t)$ oscillates — it has positive and negative values. If you want to track the **energy envelope** (how strong the signal is at any instant), you can't just take $|x(t)|$ because that misses the smooth curved shape between peaks.

**The solution — Analytic Signal:** We construct a *companion signal* $\hat{x}(t)$ (the Hilbert transform) that is always 90° out of phase with $x(t)$. Together, $x(t)$ and $\hat{x}(t)$ form the **analytic signal**:

$$z(t) = x(t) + j\hat{x}(t)$$

Think of it as a rotating phasor in the complex plane. The **magnitude** of that phasor at every moment gives you the instantaneous amplitude (envelope). The **angle** gives you the instantaneous phase.

**Why FFT-based Hilbert?**  
In the frequency domain, the Hilbert transform is simply:
- Multiply positive frequencies by $-j$ (shift phase by −90°)
- Multiply negative frequencies by $+j$ (shift phase by +90°)
- Zero frequency stays the same

This is extremely efficient to compute via FFT.


---
## 📐 Mathematical Formulation Cell 2.2

**Hilbert Transform (continuous):**

$$\hat{x}(t) = \mathcal{H}\{x(t)\} = \frac{1}{\pi} \ \text{P.V.} \int_{-\infty}^{\infty} \frac{x(\tau)}{t - \tau} \, d\tau$$

**Frequency domain implementation (what we actually use):**

$$\hat{X}(f) = -j \cdot \text{sgn}(f) \cdot X(f)$$

Where:
$$\text{sgn}(f) = \begin{cases} +1 & f > 0 \\ 0 & f = 0 \\ -1 & f < 0 \end{cases}$$

**Analytic Signal:**

$$z(t) = x(t) + j\hat{x}(t) = A(t) \cdot e^{j\phi(t)}$$

Where $A(t)$ is the instantaneous amplitude and $\phi(t)$ is the instantaneous phase.

**FFT-based algorithm (Marple, 1999):**

$$Z(f) = \begin{cases} 2 X(f) & f > 0 \\ X(f) & f = 0 \\ 0 & f < 0 \end{cases}$$

Then $z(t) = \text{IFFT}\{Z(f)\}$


In [ ]:
# ============================================================
# PART 2: Hilbert Transform from Scratch using FFT
# ============================================================

def hilbert_transform_fft(x):
    """
    Compute the Hilbert Transform and Analytic Signal from scratch.
    Uses the FFT-based method (Marple algorithm).

    Parameters:
        x : 1D numpy array (real-valued seismic trace)

    Returns:
        analytic_signal : complex array z(t) = x(t) + j*x_hat(t)
        x_hat           : Hilbert transform (quadrature component)
    """
    N = len(x)

    # Step 1: FFT of the input signal
    X = np.fft.fft(x)

    # Step 2: Build the one-sided weighting function h
    # h doubles positive frequencies, zeros negative frequencies
    h = np.zeros(N)
    if N % 2 == 0:          # Even length
        h[0] = 1            # DC component (f=0)
        h[1:N//2] = 2       # Positive frequencies × 2
        h[N//2] = 1         # Nyquist frequency
        # h[N//2+1:] = 0    # Negative frequencies (already zero)
    else:                   # Odd length
        h[0] = 1
        h[1:(N+1)//2] = 2
        # h[(N+1)//2:] = 0  # Negative frequencies

    # Step 3: Multiply FFT by h and take IFFT
    Z = X * h               # One-sided spectrum → analytic signal spectrum
    analytic_signal = np.fft.ifft(Z)   # Back to time domain (complex)

    # Step 4: Extract the Hilbert transform (imaginary part)
    x_hat = np.imag(analytic_signal)   # Quadrature component

    return analytic_signal, x_hat


# Apply to our seismic trace
analytic_signal, x_hat = hilbert_transform_fft(seismic_trace)

print("Hilbert Transform computed successfully!")
print(f"  Input  (real)   : shape={seismic_trace.shape}, dtype={seismic_trace.dtype}")
print(f"  Output (complex): shape={analytic_signal.shape}, dtype={analytic_signal.dtype}")
print(f"  Real part matches original? Max diff = "
      f"{np.max(np.abs(np.real(analytic_signal) - seismic_trace)):.2e}")


## 📊 Visualization Cell 2.3 — Hilbert Transform Components

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle('PART 2 — Hilbert Transform & Analytic Signal',
             fontsize=15, fontweight='bold', color='white')

# Original trace
ax = axes[0]
ax.plot(t*1000, seismic_trace, color='#FFD700', lw=1.8, label='x(t) — Original trace')
ax.axhline(0, color='#555', lw=0.8); ax.grid(True)
ax.set(title='Original Seismic Trace  x(t)', xlabel='Time (ms)', ylabel='Amplitude')
ax.legend(fontsize=10)

# Hilbert transform
ax = axes[1]
ax.plot(t*1000, x_hat, color='#FF69B4', lw=1.8, label='x̂(t) — Hilbert Transform')
ax.plot(t*1000, seismic_trace, color='#FFD700', lw=1, alpha=0.4, ls='--', label='x(t) for ref')
ax.axhline(0, color='#555', lw=0.8); ax.grid(True)
ax.set(title='Hilbert Transform  x̂(t)  [90° phase shifted]', xlabel='Time (ms)', ylabel='Amplitude')
ax.legend(fontsize=10)

# Complex plane view at a specific time
ax = axes[2]
ax.plot(np.real(analytic_signal), np.imag(analytic_signal),
        color='#00CED1', lw=0.8, alpha=0.6, label='Analytic signal trajectory')
# Highlight a few points
idx_pts = [100, 200, 250, 300, 375, 440]
for idx in idx_pts:
    ax.plot(np.real(analytic_signal[idx]), np.imag(analytic_signal[idx]),
            'o', color='#FF6347', ms=6)
    ax.annotate(f't={t[idx]*1000:.0f}ms',
                xy=(np.real(analytic_signal[idx]), np.imag(analytic_signal[idx])),
                fontsize=7, color='white', xytext=(5, 5), textcoords='offset points')
ax.axhline(0, color='#555', lw=0.8); ax.axvline(0, color='#555', lw=0.8)
ax.set(title='Complex Plane: z(t) = x(t) + j·x̂(t)', xlabel='Real part', ylabel='Imaginary part')
ax.legend(fontsize=10); ax.grid(True); ax.set_aspect('equal')

plt.tight_layout(); plt.show()


## 🔍 Validation Cell 2.4 — Compare with scipy.signal.hilbert

In [ ]:
# Allowed: scipy validation only
from scipy.signal import hilbert as scipy_hilbert

analytic_scipy = scipy_hilbert(seismic_trace)
x_hat_scipy    = np.imag(analytic_scipy)

# Compute difference
max_err = np.max(np.abs(x_hat - x_hat_scipy))
rms_err = np.sqrt(np.mean((x_hat - x_hat_scipy)**2))

print("=" * 55)
print("  PART 2 — HILBERT TRANSFORM VALIDATION")
print("=" * 55)
print(f"  Our result  (max) : {np.max(np.abs(x_hat)):.6f}")
print(f"  SciPy result(max) : {np.max(np.abs(x_hat_scipy)):.6f}")
print(f"  Max absolute error: {max_err:.2e}")
print(f"  RMS error         : {rms_err:.2e}")
print(f"  Result            : {'✅ PASS (error < 1e-10)' if max_err < 1e-10 else '⚠️  Check implementation'}")
print("=" * 55)

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
fig.suptitle('PART 2 — Validation: Our Hilbert vs SciPy', fontsize=14, fontweight='bold', color='white')

axes[0].plot(t*1000, x_hat, color='#FF69B4', lw=2, label='Our Hilbert')
axes[0].plot(t*1000, x_hat_scipy, color='#00FF7F', lw=1, ls='--', alpha=0.7, label='SciPy Hilbert')
axes[0].set(title='Comparison', xlabel='Time (ms)', ylabel='Amplitude')
axes[0].legend(fontsize=10); axes[0].grid(True)

axes[1].plot(t*1000, np.abs(x_hat - x_hat_scipy), color='#FF4500', lw=1.2)
axes[1].set(title='Absolute Difference (should be ~machine precision)', xlabel='Time (ms)', ylabel='|error|')
axes[1].grid(True)

plt.tight_layout(); plt.show()


---
# PART 3: Envelope (Instantaneous Amplitude)

---
## 🧠 Theory Cell 3.1 — What is the Envelope?

The **envelope** is the smooth curve that "wraps around" the peaks of the seismic trace. It tells you: *how much energy is passing through at any instant?*

**Physical meaning in seismics:**
- **High envelope** → Strong reflector (large impedance contrast between layers)
- **Low envelope** → Weak reflector or quiet zone
- **Envelope shape** → Directly related to the wavelet shape and lithology changes

The envelope is also called **Instantaneous Amplitude** or **Reflection Strength**.

**Why not just use |x(t)|?**  
$|x(t)|$ follows every zero crossing and produces a spiky result. The analytic signal magnitude gives a **smooth, physically meaningful** envelope.


---
## 📐 Mathematical Formulation Cell 3.2

Given the analytic signal $z(t) = x(t) + j\hat{x}(t)$, the **instantaneous amplitude** (envelope) is:

$$A(t) = |z(t)| = \sqrt{x(t)^2 + \hat{x}(t)^2}$$

This is simply the **modulus** of the complex analytic signal at every time step.

**Properties:**
- $A(t) \geq 0$ always
- $A(t) \geq |x(t)|$ always (envelope is always above the trace)
- $A(t)$ is smooth (no zero-crossings unless signal goes to zero)


In [ ]:
# ============================================================
# PART 3: Instantaneous Amplitude (Envelope) from Scratch
# ============================================================

def instantaneous_amplitude(analytic_sig):
    """
    Compute the envelope (instantaneous amplitude) from analytic signal.
    A(t) = |z(t)| = sqrt(x(t)^2 + x_hat(t)^2)

    Parameters:
        analytic_sig : complex analytic signal z(t)

    Returns:
        envelope : 1D real array A(t)
    """
    return np.sqrt(np.real(analytic_sig)**2 + np.imag(analytic_sig)**2)


# Compute envelope on clean and noisy traces
envelope_clean = instantaneous_amplitude(analytic_signal)

analytic_noisy, _ = hilbert_transform_fft(seismic_trace_noisy)
envelope_noisy    = instantaneous_amplitude(analytic_noisy)

print("Envelope (Instantaneous Amplitude) computed!")
print(f"  Max envelope (clean) : {envelope_clean.max():.4f}")
print(f"  Max envelope (noisy) : {envelope_noisy.max():.4f}")
print(f"  Envelope always >= 0 : {np.all(envelope_clean >= 0)}")
print(f"  Envelope >= |trace|  : {np.all(envelope_clean >= np.abs(seismic_trace) - 1e-10)}")


## 📊 Visualization Cell 3.3

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('PART 3 — Instantaneous Amplitude (Envelope)',
             fontsize=15, fontweight='bold', color='white')

# Clean trace + envelope
ax = axes[0]
ax.plot(t*1000, seismic_trace, color='#FFD700', lw=1.5, label='x(t) — Seismic trace', alpha=0.8)
ax.plot(t*1000,  envelope_clean, color='#FF4500', lw=2.2, label='A(t) — Envelope (clean)', zorder=5)
ax.plot(t*1000, -envelope_clean, color='#FF4500', lw=2.2, alpha=0.5, zorder=5)
ax.fill_between(t*1000, envelope_clean, -envelope_clean, alpha=0.1, color='#FF4500')
ax.axhline(0, color='#555', lw=0.8)
ax.set(title='Clean Trace with Envelope', xlabel='Time (ms)', ylabel='Amplitude')
ax.legend(fontsize=10); ax.grid(True)

# Noisy trace + envelope comparison
ax = axes[1]
ax.plot(t*1000, seismic_trace_noisy, color='#B0C4DE', lw=1, alpha=0.6, label='Noisy trace')
ax.plot(t*1000, envelope_noisy, color='#FF6347', lw=2, label='A(t) — Envelope (noisy)')
ax.plot(t*1000, envelope_clean, color='#FF4500', lw=2, ls='--', label='A(t) — Envelope (clean)')
ax.axhline(0, color='#555', lw=0.8)
ax.set(title='Noisy Trace with Envelope (Noise Effect)', xlabel='Time (ms)', ylabel='Amplitude')
ax.legend(fontsize=10); ax.grid(True)

plt.tight_layout(); plt.show()


## 🔍 Validation Cell 3.4

In [ ]:
# Scipy envelope for validation
envelope_scipy = np.abs(scipy_hilbert(seismic_trace))

max_err = np.max(np.abs(envelope_clean - envelope_scipy))
print("=" * 55)
print("  PART 3 — ENVELOPE VALIDATION")
print("=" * 55)
print(f"  Max error vs SciPy : {max_err:.2e}")
print(f"  Result             : {'✅ PASS' if max_err < 1e-10 else '⚠️ Check'}")
print("=" * 55)


---
# PART 4: Instantaneous Phase

---
## 🧠 Theory Cell 4.1 — What is Instantaneous Phase?

While the **envelope** tells us *how much* energy there is, the **instantaneous phase** tells us *where we are in the cycle* of the waveform at any instant.

**Intuition:** Think of a clock hand rotating. The angle of the clock hand at any moment is the "phase." For a seismic trace, phase tracks whether we're at a peak (+90°), zero crossing (0° or 180°), or trough (−90°).

**Why use it in seismics?**
- Sensitive to **waveform shape changes** even when amplitude is constant
- Highlights **lateral continuity** of reflectors in 2D sections
- Helps detect **unconformities** and **faults**
- Independent of amplitude (useful when amplitude is unreliable)

**Phase wrapping:** Phase is defined modulo 2π. As the signal oscillates, phase continuously increases but gets "wrapped" back into [−π, +π]. This creates sharp discontinuities (jumps from +π to −π) that are mathematically correct but visually confusing.


---
## 📐 Mathematical Formulation Cell 4.2

Given analytic signal $z(t) = x(t) + j\hat{x}(t)$:

**Instantaneous Phase:**

$$\phi(t) = \angle z(t) = \arctan\!2\!\left(\hat{x}(t),\, x(t)\right)$$

Where $\arctan2$ is the four-quadrant arctangent:

$$\phi(t) \in (-\pi,\ +\pi]$$

**Unwrapped Phase:**  
To remove jumps, we add/subtract $2\pi$ when a jump $> \pi$ occurs:

$$\phi_{unwrapped}(t) = \phi(t) + 2\pi \cdot k(t)$$

where $k(t)$ is the cumulative correction count.


In [ ]:
# ============================================================
# PART 4: Instantaneous Phase from Scratch
# ============================================================

def instantaneous_phase(analytic_sig):
    """
    Compute the instantaneous phase from the analytic signal.
    phi(t) = arctan2(imag(z), real(z))
    Range: (-pi, +pi]

    Parameters:
        analytic_sig : complex analytic signal

    Returns:
        phase : wrapped instantaneous phase in radians
    """
    return np.arctan2(np.imag(analytic_sig), np.real(analytic_sig))


def unwrap_phase(phase):
    """
    Unwrap phase by correcting jumps larger than pi.
    Built from scratch — no np.unwrap used.

    Parameters:
        phase : wrapped phase array (radians)

    Returns:
        unwrapped : continuously increasing phase
    """
    unwrapped = phase.copy().astype(float)
    correction = 0.0
    for i in range(1, len(phase)):
        diff = phase[i] - phase[i-1]
        # If jump > +pi, subtract 2*pi (wrapped forward)
        if diff > np.pi:
            correction -= 2 * np.pi
        # If jump < -pi, add 2*pi (wrapped backward)
        elif diff < -np.pi:
            correction += 2 * np.pi
        unwrapped[i] = phase[i] + correction
    return unwrapped


# Compute instantaneous phase
inst_phase         = instantaneous_phase(analytic_signal)
inst_phase_unwrap  = unwrap_phase(inst_phase)

# Convert to degrees for display
inst_phase_deg        = np.degrees(inst_phase)
inst_phase_unwrap_deg = np.degrees(inst_phase_unwrap)

print("Instantaneous Phase computed!")
print(f"  Wrapped range   : [{inst_phase.min():.3f}, {inst_phase.max():.3f}] rad")
print(f"  Unwrapped range : [{inst_phase_unwrap.min():.1f}, {inst_phase_unwrap.max():.1f}] rad")


## 📊 Visualization Cell 4.3

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle('PART 4 — Instantaneous Phase', fontsize=15, fontweight='bold', color='white')

# Original trace
ax = axes[0]
ax.plot(t*1000, seismic_trace, color='#FFD700', lw=1.5)
ax.axhline(0, color='#555', lw=0.8)
ax.set(title='Seismic Trace x(t)', xlabel='Time (ms)', ylabel='Amplitude'); ax.grid(True)

# Wrapped phase
ax = axes[1]
ax.plot(t*1000, inst_phase_deg, color='#DA70D6', lw=1.2, label='Wrapped phase')
ax.axhline(180,  color='#555', lw=0.5, ls=':'); ax.axhline(-180, color='#555', lw=0.5, ls=':')
ax.set(title='Instantaneous Phase — Wrapped  φ(t) ∈ (−180°, +180°]',
       xlabel='Time (ms)', ylabel='Phase (degrees)', ylim=[-200, 200])
ax.legend(fontsize=10); ax.grid(True)

# Unwrapped phase
ax = axes[2]
ax.plot(t*1000, inst_phase_unwrap_deg, color='#7FFFD4', lw=1.5, label='Unwrapped phase')
ax.set(title='Instantaneous Phase — Unwrapped  (continuous)',
       xlabel='Time (ms)', ylabel='Phase (degrees)')
ax.legend(fontsize=10); ax.grid(True)

plt.tight_layout(); plt.show()


## 🔍 Validation Cell 4.4

In [ ]:
# SciPy phase for comparison
phase_scipy   = np.angle(scipy_hilbert(seismic_trace))
phase_err     = np.max(np.abs(inst_phase - phase_scipy))

# NumPy unwrap for comparison
unwrap_numpy  = np.unwrap(inst_phase)
unwrap_err    = np.max(np.abs(inst_phase_unwrap - unwrap_numpy))

print("=" * 55)
print("  PART 4 — INSTANTANEOUS PHASE VALIDATION")
print("=" * 55)
print(f"  Phase error vs SciPy      : {phase_err:.2e}")
print(f"  Unwrap error vs np.unwrap : {unwrap_err:.2e}")
print(f"  Wrapped phase result      : {'✅ PASS' if phase_err < 1e-10 else '⚠️ Check'}")
print(f"  Unwrap result             : {'✅ PASS' if unwrap_err < 1e-10 else '⚠️ Check'}")
print("=" * 55)


---
# PART 5: Instantaneous Frequency

---
## 🧠 Theory Cell 5.1 — What is Instantaneous Frequency?

Instantaneous frequency (IF) is the **rate of change of phase** with time. It tells you: *how fast is the waveform oscillating right now?*

**Physical meaning:**
- Tracks **frequency changes within the wavelet** as it travels through different rock layers
- **Low IF** → wave slowing down (high-velocity layer? attenuation?)
- **High IF** → wave speeding up or thin-bed effects
- IF is very **noise-sensitive** — even small noise causes large phase jumps → large IF spikes

**Challenges:**
1. Numerical differentiation amplifies high-frequency noise
2. Negative frequencies (non-physical) appear when envelope is near zero
3. Requires careful smoothing or masking


---
## 📐 Mathematical Formulation Cell 5.2

**Instantaneous Frequency (continuous):**

$$f_i(t) = \frac{1}{2\pi} \frac{d\phi(t)}{dt}$$

**Discrete numerical differentiation (central difference):**

$$\frac{d\phi}{dt}\bigg|_n \approx \frac{\phi[n+1] - \phi[n-1]}{2 \Delta t}$$

**From unwrapped phase** (avoids wrap-around errors):

$$f_i[n] = \frac{\phi_{unwrapped}[n+1] - \phi_{unwrapped}[n-1]}{4\pi \Delta t}$$

**Edge handling:** Use forward/backward difference at the first and last samples.


In [ ]:
# ============================================================
# PART 5: Instantaneous Frequency from Scratch
# ============================================================

def instantaneous_frequency(phase_unwrapped, dt):
    """
    Compute instantaneous frequency from unwrapped phase
    using central difference numerical differentiation.

    f_i(t) = (1 / 2*pi) * d(phi) / dt

    Parameters:
        phase_unwrapped : unwrapped phase in radians
        dt              : sampling interval (seconds)

    Returns:
        inst_freq : instantaneous frequency in Hz
    """
    N = len(phase_unwrapped)
    inst_freq = np.zeros(N)

    # Central difference for interior points
    for n in range(1, N - 1):
        inst_freq[n] = (phase_unwrapped[n+1] - phase_unwrapped[n-1]) / (4 * np.pi * dt)

    # Forward difference at left edge
    inst_freq[0] = (phase_unwrapped[1] - phase_unwrapped[0]) / (2 * np.pi * dt)

    # Backward difference at right edge
    inst_freq[-1] = (phase_unwrapped[-1] - phase_unwrapped[-2]) / (2 * np.pi * dt)

    return inst_freq


def smooth_signal(x, window=11):
    """
    Simple moving average smoother (built from scratch).
    """
    kernel = np.ones(window) / window
    # Use np.convolve with 'same' mode
    return np.convolve(x, kernel, mode='same')


# Compute IF from unwrapped phase
inst_freq = instantaneous_frequency(inst_phase_unwrap, dt)

# Smooth to reduce noise spikes
inst_freq_smooth = smooth_signal(inst_freq, window=15)

# Mask unreliable IF values where envelope is very small
env_threshold = 0.05 * envelope_clean.max()
inst_freq_masked = inst_freq_smooth.copy()
inst_freq_masked[envelope_clean < env_threshold] = np.nan   # mask low-amplitude zones

print("Instantaneous Frequency computed!")
print(f"  Raw IF range    : [{np.nanmin(inst_freq):.1f}, {np.nanmax(inst_freq):.1f}] Hz")
print(f"  Smoothed range  : [{np.nanmin(inst_freq_smooth):.1f}, {np.nanmax(inst_freq_smooth):.1f}] Hz")
print(f"  Dominant freq   : {f_peak} Hz  (expected ~{f_peak} Hz)")


## 📊 Visualization Cell 5.3

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 13))
fig.suptitle('PART 5 — Instantaneous Frequency', fontsize=15, fontweight='bold', color='white')

ax = axes[0]
ax.plot(t*1000, seismic_trace, color='#FFD700', lw=1.5)
ax.set(title='Seismic Trace x(t)', xlabel='Time (ms)', ylabel='Amplitude'); ax.grid(True)

ax = axes[1]
ax.plot(t*1000, envelope_clean, color='#FF4500', lw=1.5)
ax.axhline(env_threshold, color='#888', lw=1, ls='--', label=f'Threshold = {env_threshold:.3f}')
ax.set(title='Envelope A(t) with Masking Threshold', xlabel='Time (ms)', ylabel='Amplitude')
ax.legend(fontsize=9); ax.grid(True)

ax = axes[2]
ax.plot(t*1000, inst_freq, color='#98FB98', lw=0.8, alpha=0.5, label='Raw IF')
ax.plot(t*1000, inst_freq_smooth, color='#00CED1', lw=1.8, label='Smoothed IF')
ax.axhline(f_peak, color='#FF69B4', lw=1.5, ls='--', label=f'Expected = {f_peak} Hz')
ax.set(title='Instantaneous Frequency (Raw vs Smoothed)', xlabel='Time (ms)', ylabel='Freq (Hz)', ylim=[-20, 100])
ax.legend(fontsize=9); ax.grid(True)

ax = axes[3]
ax.plot(t*1000, inst_freq_masked, color='#00CED1', lw=1.8, label='IF (amplitude-masked)')
ax.axhline(f_peak, color='#FF69B4', lw=1.5, ls='--', label=f'Expected = {f_peak} Hz')
ax.set(title='Instantaneous Frequency (Masked: NaN where envelope is weak)',
       xlabel='Time (ms)', ylabel='Freq (Hz)', ylim=[-10, 80])
ax.legend(fontsize=9); ax.grid(True)

plt.tight_layout(); plt.show()


## 🔍 Validation Cell 5.4

In [ ]:
# Compare IF with numerical derivative of np.unwrap
phase_scipy_unwrap = np.unwrap(phase_scipy)
inst_freq_scipy    = np.diff(phase_scipy_unwrap) / (2 * np.pi * dt)
inst_freq_scipy_full = np.append(inst_freq_scipy, inst_freq_scipy[-1])

mean_our   = np.nanmean(inst_freq[envelope_clean > env_threshold])
mean_scipy = np.nanmean(inst_freq_scipy_full[envelope_clean > env_threshold])

print("=" * 55)
print("  PART 5 — INSTANTANEOUS FREQUENCY VALIDATION")
print("=" * 55)
print(f"  Our mean IF (masked)   : {mean_our:.2f} Hz")
print(f"  SciPy-based mean IF    : {mean_scipy:.2f} Hz")
print(f"  Expected peak freq     : {f_peak} Hz")
print(f"  Within 5 Hz of expected: {'✅ PASS' if abs(mean_our - f_peak) < 5 else '⚠️ Check'}")
print("=" * 55)


---
# PART 6: RMS Amplitude and Energy Attributes

---
## 🧠 Theory Cell 6.1 — RMS Amplitude

**RMS (Root Mean Square) Amplitude** is one of the simplest yet most powerful seismic attributes. Instead of looking at a single sample, it measures the **average energy in a sliding window** around each sample.

**Why RMS?**
- Robust against noise (averages out random fluctuations)
- Directly related to **acoustic impedance** and **fluid content**
- Used in **bright spot detection** (gas sands cause high amplitude anomalies)
- AVO (Amplitude Versus Offset) analysis uses RMS

**Window choice matters:**
- Too small → noisy, sensitive to individual samples
- Too large → smears geology, loses resolution
- Typical: half-window of 20–50 ms for 30 Hz data


---
## 📐 Mathematical Formulation Cell 6.2

**RMS Amplitude** in a window of length $2M+1$ centered at sample $n$:

$$A_{RMS}[n] = \sqrt{\frac{1}{2M+1} \sum_{k=n-M}^{n+M} x[k]^2}$$

**Energy Attribute** (unnormalized):

$$E[n] = \sum_{k=n-M}^{n+M} x[k]^2$$

Where:
- $M$ → half-window length in samples
- $x[k]$ → seismic amplitude at sample $k$
- Edge handling: use partial windows at trace start/end


In [ ]:
# ============================================================
# PART 6: RMS Amplitude and Energy from Scratch
# ============================================================

def rms_amplitude(trace, half_window):
    """
    Compute RMS amplitude using a sliding window.
    Built from scratch — no np.convolve shortcut for the core logic.

    Parameters:
        trace       : 1D seismic trace
        half_window : half-length of the sliding window (samples)

    Returns:
        rms : RMS amplitude at each sample
    """
    N = len(trace)
    rms = np.zeros(N)

    for n in range(N):
        # Define window bounds with edge clamping
        start = max(0, n - half_window)
        end   = min(N, n + half_window + 1)
        window_samples = trace[start:end]
        # RMS formula
        rms[n] = np.sqrt(np.mean(window_samples**2))

    return rms


def energy_attribute(trace, half_window):
    """
    Compute sliding window energy (sum of squared amplitudes).

    Parameters:
        trace       : 1D seismic trace
        half_window : half-length of sliding window

    Returns:
        energy : energy at each sample
    """
    N = len(trace)
    energy = np.zeros(N)

    for n in range(N):
        start = max(0, n - half_window)
        end   = min(N, n + half_window + 1)
        energy[n] = np.sum(trace[start:end]**2)

    return energy


# Compute with multiple window sizes to show effect
hw_small  = int(0.020 / dt)   # 20 ms half-window
hw_medium = int(0.050 / dt)   # 50 ms half-window
hw_large  = int(0.100 / dt)   # 100 ms half-window

rms_small  = rms_amplitude(seismic_trace, hw_small)
rms_medium = rms_amplitude(seismic_trace, hw_medium)
rms_large  = rms_amplitude(seismic_trace, hw_large)
energy_med = energy_attribute(seismic_trace, hw_medium)

print("RMS Amplitude computed for multiple window sizes!")
print(f"  Window small  ({hw_small*2+1} samples, {hw_small*2*dt*1000:.0f} ms) max RMS: {rms_small.max():.4f}")
print(f"  Window medium ({hw_medium*2+1} samples, {hw_medium*2*dt*1000:.0f} ms) max RMS: {rms_medium.max():.4f}")
print(f"  Window large  ({hw_large*2+1} samples, {hw_large*2*dt*1000:.0f} ms) max RMS: {rms_large.max():.4f}")


## 📊 Visualization Cell 6.3

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle('PART 6 — RMS Amplitude & Energy Attribute',
             fontsize=15, fontweight='bold', color='white')

ax = axes[0]
ax.plot(t*1000, seismic_trace, color='#FFD700', lw=1.5, label='Seismic trace', alpha=0.7)
ax.plot(t*1000, rms_small,  color='#FF4500', lw=1.5, label=f'RMS ({hw_small*2*dt*1000:.0f} ms win)')
ax.plot(t*1000, rms_medium, color='#00CED1', lw=1.8, label=f'RMS ({hw_medium*2*dt*1000:.0f} ms win)')
ax.plot(t*1000, rms_large,  color='#DA70D6', lw=1.5, label=f'RMS ({hw_large*2*dt*1000:.0f} ms win)')
ax.axhline(0, color='#555', lw=0.8)
ax.set(title='RMS Amplitude — Effect of Window Size', xlabel='Time (ms)', ylabel='Amplitude')
ax.legend(fontsize=9); ax.grid(True)

ax = axes[1]
ax.plot(t*1000, envelope_clean, color='#FF4500', lw=2, label='Envelope A(t)', ls='--')
ax.plot(t*1000, rms_medium, color='#00CED1', lw=2, label=f'RMS (medium window)')
ax.set(title='Envelope vs RMS Amplitude (Both measure "local energy")',
       xlabel='Time (ms)', ylabel='Amplitude')
ax.legend(fontsize=9); ax.grid(True)

ax = axes[2]
ax.fill_between(t*1000, energy_med, alpha=0.4, color='#9370DB')
ax.plot(t*1000, energy_med, color='#9370DB', lw=1.5, label='Energy attribute')
ax.set(title='Energy Attribute (Sliding Window Sum of Squares)',
       xlabel='Time (ms)', ylabel='Energy')
ax.legend(fontsize=9); ax.grid(True)

plt.tight_layout(); plt.show()


## 🔍 Validation Cell 6.4

In [ ]:
# Compare against numpy vectorized computation
def rms_numpy_validation(trace, half_window):
    """Vectorized reference using np.convolve for speed comparison."""
    N = len(trace)
    x2 = trace**2
    win_size = 2 * half_window + 1
    cumsum = np.cumsum(np.pad(x2, (half_window, half_window), mode='edge'))
    window_sums = cumsum[win_size:] - cumsum[:N]
    return np.sqrt(window_sums[:N] / win_size)

rms_ref = rms_numpy_validation(seismic_trace, hw_medium)
max_err = np.max(np.abs(rms_medium - rms_ref[:len(rms_medium)]))

print("=" * 55)
print("  PART 6 — RMS VALIDATION")
print("=" * 55)
print(f"  Max error vs vectorized: {max_err:.4f}")
print(f"  Result: {'✅ PASS (edge rounding OK)' if max_err < 0.01 else '⚠️ Check edges'}")
print("  Note: Small edge differences are expected due to")
print("  different edge-padding strategies.")
print("=" * 55)


---
# PART 7: Apply on 2D Seismic Section

---
## 🧠 Theory Cell 7.1 — From 1D Trace to 2D Section

A **2D seismic section** is simply a collection of 1D traces placed side by side:

- **Rows** → time samples (depth direction)
- **Columns** → trace index (lateral / offset direction)

Each column is one seismic trace. To compute seismic attributes for the whole section, we apply our 1D functions **column by column**.

**Why 2D matters:**
- Reveals **lateral continuity** of geological horizons
- Shows **fault zones** (breaks in continuity of phase/amplitude)
- Enables **stratigraphic interpretation**
- Instantaneous phase is particularly powerful for showing layer continuity


In [ ]:
# ============================================================
# PART 7: 2D Seismic Section — Build & Attribute Extraction
# ============================================================

# ── Build a synthetic 2D section ──
n_traces = 60   # Number of traces (lateral direction)

# Each trace has slightly varying reflector times (simulate a dipping layer)
def build_2d_section(t, n_traces, f_peak, dt):
    """
    Build a synthetic 2D seismic section.
    Reflectors dip gently across traces.
    """
    N = len(t)
    section = np.zeros((N, n_traces))

    t_wav = np.arange(-0.1, 0.1, dt)
    wav   = ricker_wavelet(t_wav, f_peak)

    # Base reflector times (slightly dipping + one fault)
    base_refs = np.array([0.20, 0.42, 0.60, 0.78])

    for tr in range(n_traces):
        # Add linear dip to each reflector
        dip = tr * 0.001    # 1 ms per trace lateral dip

        # Introduce a small fault offset after trace 30
        fault_offset = 0.025 if tr > 30 else 0.0

        reflectivity = np.zeros(N)
        amps = [0.9, -0.7, 1.0, -0.5]

        for ref_t, amp in zip(base_refs, amps):
            t_shifted = ref_t + dip + fault_offset
            idx = int(t_shifted / dt)
            if 0 <= idx < N:
                reflectivity[idx] = amp

        # Add noise (varying SNR across section)
        np.random.seed(tr)
        noise_amp = 0.04 + 0.01 * np.sin(tr * 0.2)
        section[:, tr] = np.convolve(reflectivity, wav, mode='same')                          + noise_amp * np.random.randn(N)

    return section

section_2d = build_2d_section(t, n_traces, f_peak, dt)
print(f"2D Seismic Section shape: {section_2d.shape}  (samples × traces)")
print(f"  Rows    = {section_2d.shape[0]} time samples ({t_max*1000:.0f} ms)")
print(f"  Columns = {section_2d.shape[1]} traces")


In [ ]:
# ── Apply all attributes trace-by-trace ──
def apply_attribute_2d(section, attr_func, **kwargs):
    """
    Apply a 1D attribute function to every trace of a 2D section.

    Parameters:
        section   : 2D array (n_samples × n_traces)
        attr_func : function that takes a 1D trace and returns 1D attribute
        **kwargs  : extra arguments passed to attr_func

    Returns:
        attr_section : 2D attribute section
    """
    N, n_tr = section.shape
    attr_section = np.zeros((N, n_tr))

    for tr in range(n_tr):
        trace = section[:, tr]
        attr_section[:, tr] = attr_func(trace, **kwargs)

    return attr_section


# Wrapper functions for 2D application
def get_envelope_1d(trace):
    analytic, _ = hilbert_transform_fft(trace)
    return instantaneous_amplitude(analytic)

def get_phase_1d(trace):
    analytic, _ = hilbert_transform_fft(trace)
    return instantaneous_phase(analytic)

def get_freq_1d(trace):
    analytic, _ = hilbert_transform_fft(trace)
    phase = instantaneous_phase(analytic)
    unwrapped = unwrap_phase(phase)
    freq = instantaneous_frequency(unwrapped, dt)
    env = instantaneous_amplitude(analytic)
    thresh = 0.05 * env.max()
    freq[env < thresh] = np.nan
    freq = smooth_signal(np.nan_to_num(freq), window=9)
    return freq

def get_rms_1d(trace, half_window=25):
    return rms_amplitude(trace, half_window)


# Compute all attributes for 2D section
print("Computing 2D attributes (this may take a moment)...")
section_envelope = apply_attribute_2d(section_2d, get_envelope_1d)
section_phase    = apply_attribute_2d(section_2d, get_phase_1d)
section_freq     = apply_attribute_2d(section_2d, get_freq_1d)
section_rms      = apply_attribute_2d(section_2d, get_rms_1d, half_window=25)

print("✅ All 2D attributes computed!")
print(f"   Envelope  shape: {section_envelope.shape}")
print(f"   Phase     shape: {section_phase.shape}")
print(f"   Frequency shape: {section_freq.shape}")
print(f"   RMS       shape: {section_rms.shape}")


## 📊 Visualization Cell 7.2 — 2D Section and Attributes

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('PART 7 — 2D Seismic Section & Attributes',
             fontsize=16, fontweight='bold', color='white')

extent = [0, n_traces, t_max*1000, 0]   # [left, right, bottom, top] in ms

# Original section (wiggle-style using imshow)
ax = axes[0, 0]
vmax = np.percentile(np.abs(section_2d), 98)
im = ax.imshow(section_2d, aspect='auto', cmap='RdBu_r', extent=extent,
               vmin=-vmax, vmax=vmax)
ax.set(title='Seismic Section', xlabel='Trace #', ylabel='Time (ms)')
plt.colorbar(im, ax=ax, fraction=0.03, label='Amplitude')

# Envelope
ax = axes[0, 1]
im = ax.imshow(section_envelope, aspect='auto', cmap='hot', extent=extent,
               vmin=0, vmax=np.percentile(section_envelope, 98))
ax.set(title='Envelope (Instantaneous Amplitude)', xlabel='Trace #', ylabel='Time (ms)')
plt.colorbar(im, ax=ax, fraction=0.03, label='Amplitude')

# Instantaneous Phase
ax = axes[0, 2]
im = ax.imshow(section_phase, aspect='auto', cmap='hsv', extent=extent,
               vmin=-np.pi, vmax=np.pi)
ax.set(title='Instantaneous Phase', xlabel='Trace #', ylabel='Time (ms)')
plt.colorbar(im, ax=ax, fraction=0.03, label='Phase (rad)')

# Instantaneous Frequency
ax = axes[1, 0]
freq_2d_plot = np.clip(section_freq, 0, 80)
im = ax.imshow(freq_2d_plot, aspect='auto', cmap='plasma', extent=extent)
ax.set(title='Instantaneous Frequency', xlabel='Trace #', ylabel='Time (ms)')
plt.colorbar(im, ax=ax, fraction=0.03, label='Freq (Hz)')

# RMS Amplitude
ax = axes[1, 1]
im = ax.imshow(section_rms, aspect='auto', cmap='viridis', extent=extent,
               vmin=0, vmax=np.percentile(section_rms, 98))
ax.set(title='RMS Amplitude', xlabel='Trace #', ylabel='Time (ms)')
plt.colorbar(im, ax=ax, fraction=0.03, label='RMS')

# Side-by-side single trace attributes
ax = axes[1, 2]
tr_idx = 25
ax.plot(seismic_trace if False else section_2d[:, tr_idx] / np.max(np.abs(section_2d[:, tr_idx])),
        t*1000, color='#FFD700', lw=1.2, label='Trace (norm.)')
ax.plot(section_envelope[:, tr_idx] / section_envelope[:, tr_idx].max(),
        t*1000, color='#FF4500', lw=1.8, label='Envelope (norm.)')
ax.axhline(0, color='#555', lw=0.5)
ax.set(title=f'Single Trace #{tr_idx} — Trace + Envelope',
       xlabel='Normalized Amplitude', ylabel='Time (ms)')
ax.invert_yaxis()
ax.legend(fontsize=8); ax.grid(True)

plt.tight_layout(); plt.show()


---
# PART 8: Final Comparison, Insights & Discussion

---
## 🧠 Theory Cell 8.1 — Comparing All Attributes

Now we bring everything together and critically compare all attributes on the **same trace and section**. This is the most important step for geophysical interpretation.

**Summary of attributes and their uses:**

| Attribute | Physical Meaning | Sensitive To | Common Use |
|---|---|---|---|
| **Envelope** | Signal energy/strength | Impedance contrast | Bright spots, gas sands |
| **Inst. Phase** | Position in cycle | Continuity | Faults, unconformities |
| **Inst. Frequency** | Rate of phase change | Attenuation, thin beds | Shadow zones, DHI |
| **RMS Amplitude** | Average energy | General amplitude | Hydrocarbon indicators |


In [ ]:
# ============================================================
# PART 8: Final Comparison on Single Trace
# ============================================================

fig, axes = plt.subplots(5, 1, figsize=(15, 16), sharex=True)
fig.suptitle('PART 8 — All Seismic Attributes: Final Comparison',
             fontsize=16, fontweight='bold', color='white', y=1.01)

colors = ['#FFD700', '#FF4500', '#DA70D6', '#00CED1', '#98FB98']
labels = ['Seismic Trace x(t)', 'Envelope A(t)', 'Inst. Phase φ(t)',
          'Inst. Frequency f_i(t)', 'RMS Amplitude']
data   = [seismic_trace, envelope_clean,
          inst_phase_deg, inst_freq_masked, rms_medium]
yunits = ['Amplitude', 'Amplitude', 'Phase (°)', 'Freq (Hz)', 'RMS']

for i, (ax, d, lbl, col, yu) in enumerate(zip(axes, data, labels, colors, yunits)):
    ax.plot(t*1000, d, color=col, lw=1.6, label=lbl)
    if i == 0:
        ax.fill_between(t*1000, d, 0, where=(d>0), alpha=0.2, color=col)
        ax.fill_between(t*1000, d, 0, where=(d<0), alpha=0.2, color='#FF6347')
    elif i == 1:
        ax.fill_between(t*1000, d, alpha=0.2, color=col)
    ax.axhline(0, color='#444', lw=0.8)
    ax.set_ylabel(yu, fontsize=9)
    ax.legend(loc='upper right', fontsize=9, framealpha=0.3)
    ax.grid(True)
    # Mark reflector times
    for t_ref in reflector_times:
        ax.axvline(t_ref*1000, color='white', lw=0.5, ls=':', alpha=0.4)

axes[-1].set_xlabel('Time (ms)', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# ── Noise Sensitivity Analysis ──
print("=" * 60)
print("  PART 8 — NOISE SENSITIVITY ANALYSIS")
print("=" * 60)

noise_levels = [0.01, 0.05, 0.10, 0.20]
results = {}

for nl in noise_levels:
    np.random.seed(99)
    noisy = seismic_trace + nl * np.random.randn(N)
    an, _ = hilbert_transform_fft(noisy)
    env   = instantaneous_amplitude(an)
    ph    = instantaneous_phase(an)
    ph_uw = unwrap_phase(ph)
    freq  = instantaneous_frequency(ph_uw, dt)
    rms   = rms_amplitude(noisy, hw_medium)

    env_err  = np.sqrt(np.mean((env - envelope_clean)**2))
    ph_err   = np.sqrt(np.mean((ph  - inst_phase)**2))
    freq_err = np.sqrt(np.mean((freq - instantaneous_frequency(inst_phase_unwrap, dt))**2))
    rms_err  = np.sqrt(np.mean((rms - rms_medium)**2))

    results[nl] = (env_err, ph_err, freq_err, rms_err)
    print(f"  Noise={nl:.2f} | Env RMSE={env_err:.4f} | Phase RMSE={ph_err:.4f} "
          f"| Freq RMSE={freq_err:.2f} | RMS RMSE={rms_err:.4f}")

print("=" * 60)
print("  Observation: Instantaneous Frequency is MOST sensitive to noise.")
print("  Envelope and RMS are MORE ROBUST to noise.")
print("=" * 60)


## 📊 Visualization Cell 8.2 — Noise Sensitivity

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('PART 8 — Attribute Noise Sensitivity (RMSE vs Noise Level)',
             fontsize=13, fontweight='bold', color='white')

attr_names = ['Envelope', 'Phase', 'Inst. Freq', 'RMS']
attr_colors = ['#FF4500', '#DA70D6', '#00CED1', '#98FB98']
x_vals = list(results.keys())

for i, (name, col) in enumerate(zip(attr_names, attr_colors)):
    y_vals = [results[nl][i] for nl in x_vals]
    ax.plot(x_vals, y_vals, 'o-', color=col, lw=2, ms=8, label=name)

ax.set(xlabel='Noise Level (std)', ylabel='RMSE',
       title='Noise Sensitivity: Lower = More Robust')
ax.legend(fontsize=10, framealpha=0.4); ax.grid(True)
plt.tight_layout(); plt.show()


## 📊 Visualization Cell 8.3 — All 2D Attributes Final Dashboard

In [ ]:
fig = plt.figure(figsize=(20, 14))
fig.suptitle('PART 8 — Final 2D Seismic Attribute Dashboard',
             fontsize=17, fontweight='bold', color='white', y=1.01)

extent = [0, n_traces, t_max*1000, 0]
configs = [
    (section_2d,        'RdBu_r',  'Seismic Section',           None,     None),
    (section_envelope,  'hot',     'Envelope',                  0,        None),
    (section_phase,     'hsv',     'Instantaneous Phase',       -np.pi,   np.pi),
    (np.clip(section_freq,0,80),
                        'plasma',  'Instantaneous Frequency',   None,     None),
    (section_rms,       'viridis', 'RMS Amplitude',             0,        None),
]

for i, (data2d, cmap, title, vmin, vmax) in enumerate(configs):
    ax = fig.add_subplot(2, 3, i+1)
    if vmin is None and vmax is None:
        p98 = np.nanpercentile(np.abs(data2d), 98)
        if data2d.min() < 0:
            kw = dict(vmin=-p98, vmax=p98)
        else:
            kw = dict(vmin=0, vmax=p98)
    else:
        kw = dict(vmin=vmin, vmax=vmax)
    im = ax.imshow(data2d, aspect='auto', cmap=cmap, extent=extent, **kw)
    ax.set(title=title, xlabel='Trace #', ylabel='Time (ms)')
    plt.colorbar(im, ax=ax, fraction=0.025)

# Add interpretation notes as 6th panel
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis('off')
notes = [
    "📌 INTERPRETATION GUIDE",
    "",
    "🔴 Envelope:  High values = strong",
    "   reflectors / potential gas sands",
    "",
    "🎨 Phase:  Lateral continuity shows",
    "   geological layering. Breaks =",
    "   faults or unconformities.",
    "",
    "🔵 Frequency:  Low IF zones may",
    "   indicate attenuation (gas?)",
    "   or tuning/thin-bed effects.",
    "",
    "🟢 RMS:  Smooth energy estimate.",
    "   Useful for horizon-based AVO.",
    "",
    "⚡ Note: All attributes built",
    "   from scratch using FFT-based",
    "   Hilbert Transform!",
]
for j, line in enumerate(notes):
    ax6.text(0.05, 0.95 - j * 0.057, line,
             transform=ax6.transAxes,
             fontsize=9.5, color='white', va='top',
             fontfamily='monospace')

plt.tight_layout()
plt.show()


---
## 🎯 Final Summary — What You Built from Scratch

```
╔══════════════════════════════════════════════════════════════╗
║         SEISMIC ATTRIBUTES FROM SCRATCH — SUMMARY           ║
╠══════════════════════════════════════════════════════════════╣
║  PART 1 ✅  Seismic trace, Ricker wavelet, convolutional     ║
║             model, frequency spectrum                        ║
║                                                              ║
║  PART 2 ✅  Hilbert Transform via FFT (Marple algorithm)     ║
║             Analytic signal z(t) = x(t) + j·x̂(t)          ║
║                                                              ║
║  PART 3 ✅  Envelope A(t) = |z(t)|                          ║
║                                                              ║
║  PART 4 ✅  Instantaneous Phase φ(t) = arctan2(x̂, x)       ║
║             Phase unwrapping from scratch                    ║
║                                                              ║
║  PART 5 ✅  Instantaneous Frequency f_i = dφ/dt / 2π        ║
║             Central difference + amplitude masking          ║
║                                                              ║
║  PART 6 ✅  RMS Amplitude — sliding window                   ║
║             Energy attribute                                 ║
║                                                              ║
║  PART 7 ✅  2D Section — trace-by-trace application          ║
║             Synthetic section with dip + fault               ║
║                                                              ║
║  PART 8 ✅  Final dashboard, noise sensitivity analysis      ║
╚══════════════════════════════════════════════════════════════╝

  Key Takeaway: Everything flows from the Hilbert Transform!
  Analytic Signal → Envelope → Phase → Frequency
```
